# Sentiment classifier

Movie reviews sentiment classification.

3 possible categories: positive, negative or unclear.

English language.

In [ ]:
import ollama

# Initial version

In [2]:
model = "phi4-mini"

You can provide different roles to the model: user, system, etc.
- **System** (for developer): context for the chatbot, personality, red lines that should not cross (e.g. don't talk about finance)
- **User** (for end user): a human talking to the chatbot

In [3]:
system_prompt = """
    You are a ternary classifier specialized in sentiment detection. You have been trained on a dataset of movie reviews 
    and you have been deployed to a movie review website. Your task is to classify the sentiment of the reviews as positive (reserved word [POSITIVE]), negative (reserved word [NEGATIVE]) or unclear (reserved word [UNK]).
    Remember that you are a ternary classifier, so you can only output one of the three reserved words and you should not output anything else, not even an explanation. Otherwise a kitty will die.
    """

In [4]:
user_prompt_list = ["I loved the movie, it was amazing!",
                     "I hated the movie, it was terrible!",
                     "The movie was okay, I guess.", 
                     "I don't know how I feel about the movie.", 
                     "I don't know how I feel about the movie, but the acting was great."]

In [5]:
response = ollama.chat(model=model, messages=[
        {
            'role': 'system',
            'content': system_prompt,
        },
        {
            'role': 'user',
            'content': user_prompt_list[0]
        }
    ],
    options={
        "seed": 42
    },
    stream=True)

print(user_prompt_list[0])

for chunk in response:
    print(chunk['message']['content'], end='', flush=True)

I loved the movie, it was amazing!
[POSITIVE]

Now let's run all the examples

In [6]:
for user_prompt in user_prompt_list:

    response = ollama.chat(model=model, messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ],
        options={
            "seed": 42
        },
        stream=True)

    print("Sentence> " + user_prompt)
    print("Classification> ", end='')
    for chunk in response:
        print(chunk['message']['content'], end='', flush=True)
    print("\n")

Sentence> I loved the movie, it was amazing!
Classification> [POSITIVE]

Sentence> I hated the movie, it was terrible!
Classification> [NEGATIVE]

Sentence> The movie was okay, I guess.
Classification> [UNK]

Sentence> I don't know how I feel about the movie.
Classification> [UNK]

Sentence> I don't know how I feel about the movie, but the acting was great.
Classification> POSITIVE



# Few-shot prompting

Few-shot prompting for dummies: add examples of typical questions and typical expected answers. Include the output format as well ;-)

Note: pay attention to the previous unknown examples

In [7]:
system_prompt_fewshot = """
    You are a ternary classifier specialized in sentiment detection. You have been trained on a dataset of movie reviews 
    and you have been deployed to a movie review website. Your task is to classify the sentiment of the reviews as positive (reserved word [POSITIVE]), negative (reserved word [NEGATIVE]) or unclear (reserved word [UNK]).
    Remember that you are a ternary classifier, so you can only output one of the three reserved words and you should not output anything else, not even an explanation. Otherwise a kitty will die.
    
    Some examples of reviews:
    - I loved the movie, it was amazing! [POSITIVE]
    - I hated the movie, it was terrible! [NEGATIVE]
    - The movie was okay, I guess. [POSITIVE]
    - I don't know how I feel about the movie. [UNK]
    - It was a good time [POSITIVE]
    - I did not like it all [NEGATIVE]
    - Give me my money back [NEGATIVE]
    """

In [8]:
for user_prompt in user_prompt_list:

    response = ollama.chat(model=model, messages=[
            {
                'role': 'system',
                'content': system_prompt_fewshot,
            },
            {
                'role': 'user',
                'content': user_prompt
            }
        ],
        options={
            "seed": 42
        },
        stream=True)

    print("Sentence> " + user_prompt)
    print("Classification> ", end='')
    for chunk in response:
        print(chunk['message']['content'], end='', flush=True)
    print("\n")

Sentence> I loved the movie, it was amazing!
Classification> [POSITIVE]

Sentence> I hated the movie, it was terrible!
Classification> [NEGATIVE]

Sentence> The movie was okay, I guess.
Classification> [POSITIVE]

Sentence> I don't know how I feel about the movie.
Classification> [UNK]

Sentence> I don't know how I feel about the movie, but the acting was great.
Classification> [UNK]



Want to know more tricks? Here it is an excelent guide 
https://www.promptingguide.ai/

TIP: Change-of-Thought is likely the best technique you must learn

## More difficult reviews and multi language

Phi 4 increased multilingual support compared to its predecessor (Phi 3 had not support) but it's mainly based in english. Let's challenge it:
- Other language
- Longer review

In [9]:
review = "La película era un tostón sobre multiversos que nadie entendía, pero al final se salva porque aparece Robert Downey Junior y todos aplaudieron."

messages=[
        {
            'role': 'system',
            'content': system_prompt_fewshot,
        },
        {
            'role': 'user',
            'content': review
        }
    ]

In [10]:
response = ollama.chat(model=model,
    messages=messages,
    options={
        "seed": 42
    },
    stream=True)

print("Sentence> " + review)
print("Classification> ", end='')
for chunk in response:
    print(chunk['message']['content'], end='', flush=True)
print("\n")

Sentence> La película era un tostón sobre multiversos que nadie entendía, pero al final se salva porque aparece Robert Downey Junior y todos aplaudieron.
Classification> [POSITIVE]



# Tuning the temperature

It's a parameter that controls randomness

- Creativity? Conversations? Ideas? -> High temperature
- Facts? Classifiers or any discriminative NLP task? -> Low temperature

Valid values: between 0.0 and 1.0

### Low temperature

In [11]:
response = ollama.chat(model=model,
    messages=messages,
    options={
        "seed": 42,
        "temperature": 0.1
    },
    stream=True)

print("Sentence> " + review)
print("Classification> ", end='')
for chunk in response:
    print(chunk['message']['content'], end='', flush=True)
print("\n")

Sentence> La película era un tostón sobre multiversos que nadie entendía, pero al final se salva porque aparece Robert Downey Junior y todos aplaudieron.
Classification> [POSITIVE]



### High temperature

In [12]:
response = ollama.chat(model=model,
    messages=messages,
    options={
        "seed": 42,
        "temperature": 1.0
    },
    stream=True)

print("Sentence> " + review)
print("Classification> ", end='')
for chunk in response:
    print(chunk['message']['content'], end='', flush=True)
print("\n")

Sentence> La película era un tostón sobre multiversos que nadie entendía, pero al final se salva porque aparece Robert Downey Junior y todos aplaudieron.
Classification> [POSITIVE]



Why no change? haha, few-shot prompting is very powerful.

# Long context: private information

A local LLM is very interesting to deal with private information and questions you don't want to be registered by 3rd parties.

In [13]:
system_prompt_search = """
    You are a powerful search engine. You will help me to find relevant information from documents that I will provide you.
    It's important that if you don't have enough information don't try to guess it, just say "I don't know". Information must be accurate, real
    and inside the document. If you have information related but it's not in the document, please don't include it.
"""

In [14]:
document = """

    MOTORCYCLE SALE AGREEMENT

    In Madrid, on September 8th, 2024.

    PARTIES
    On one side, Mr. Juan Martínez García, of legal age, residing at Calle Gran Vía 123, 28013, Madrid, with NIF 12345678A.
    On the other side, Ms. Laura Fernández Pérez, of legal age, residing at Calle Alcalá 456, 28014, Madrid, with NIF 87654321B.

    STATEMENTS
    Both parties agree to formalize the sale of the motorcycle described below:

        Brand: Yamaha
        Model: MT-07
        License plate: 1234ABC
        VIN: VYZ23456789
        Year of manufacture: 2020
        Mileage: 15,000 km
        ITV valid until: October 15th, 2025
        Other details: a helmet and security lock are included.

    CLAUSES

    First: Subject of the agreement
    The Seller sells, and the Buyer purchases the motorcycle described above.

    Second: Sale price
    The total sale price is 6,500 €, which has been paid by the Buyer to the Seller at the time of signing via bank transfer.

    Third: Delivery and condition of the motorcycle
    The Seller delivers the motorcycle described in this agreement at the time of signing, along with the following documents:

        Registration certificate.
        Technical inspection card.
        Receipt of paid road tax up to date.
        The Buyer declares having inspected the condition of the motorcycle and accepts it as is, releasing the Seller from any hidden defects that may appear after signing.

    Fourth: Procedures and expenses
    Both parties agree that the costs for the title transfer and any other administrative procedures arising from this sale will be borne by the Buyer.

    To formalize this agreement, both parties sign two copies of this document on the date and place indicated.

    Signatures:

    Juan Martínez García (Seller)

    Laura Fernández Pérez (Buyer)
"""

In [15]:
questions = f"""
    What is the item that is being sold? At what price and when is the sale?
"""

In [16]:
user_prompt = f"""

<Question>
{questions}
</Question>

<Document>
{document}
</Document>
"""

In [17]:
print(user_prompt)



<Question>

    What is the item that is being sold? At what price and when is the sale?

</Question>

<Document>


    MOTORCYCLE SALE AGREEMENT

    In Madrid, on September 8th, 2024.

    PARTIES
    On one side, Mr. Juan Martínez García, of legal age, residing at Calle Gran Vía 123, 28013, Madrid, with NIF 12345678A.
    On the other side, Ms. Laura Fernández Pérez, of legal age, residing at Calle Alcalá 456, 28014, Madrid, with NIF 87654321B.

    STATEMENTS
    Both parties agree to formalize the sale of the motorcycle described below:

        Brand: Yamaha
        Model: MT-07
        License plate: 1234ABC
        VIN: VYZ23456789
        Year of manufacture: 2020
        Mileage: 15,000 km
        ITV valid until: October 15th, 2025
        Other details: a helmet and security lock are included.

    CLAUSES

    First: Subject of the agreement
    The Seller sells, and the Buyer purchases the motorcycle described above.

    Second: Sale price
    The total sale price is 6

In [18]:
response = ollama.chat(model=model, messages=[
        {
            'role': 'system',
            'content': system_prompt_search,
        },
        {
            'role': 'user',
            'content': user_prompt
        }
    ],
    options={
        "seed": 42,
        "temperature": 0.9 # prev value 0.1
    },
    stream=True)

print(user_prompt)
for chunk in response:
    print(chunk['message']['content'], end='', flush=True)
print("\n")



<Question>

    What is the item that is being sold? At what price and when is the sale?

</Question>

<Document>


    MOTORCYCLE SALE AGREEMENT

    In Madrid, on September 8th, 2024.

    PARTIES
    On one side, Mr. Juan Martínez García, of legal age, residing at Calle Gran Vía 123, 28013, Madrid, with NIF 12345678A.
    On the other side, Ms. Laura Fernández Pérez, of legal age, residing at Calle Alcalá 456, 28014, Madrid, with NIF 87654321B.

    STATEMENTS
    Both parties agree to formalize the sale of the motorcycle described below:

        Brand: Yamaha
        Model: MT-07
        License plate: 1234ABC
        VIN: VYZ23456789
        Year of manufacture: 2020
        Mileage: 15,000 km
        ITV valid until: October 15th, 2025
        Other details: a helmet and security lock are included.

    CLAUSES

    First: Subject of the agreement
    The Seller sells, and the Buyer purchases the motorcycle described above.

    Second: Sale price
    The total sale price is 6

A little bit verbose (I don't care about its knowledge cutoff). This is what happens when you increase temperature and don't provide a strict output format.

## Longer context?

What if I have longer contexts? Like... many documents or a database?

Solution: RAG architecture